# 2D-to-3D — TripoSR on Colab T4

**Run All** twice: 1st installs + restarts, 2nd generates.

Runtime → **T4 GPU** | ~1GB model | Fast | Produces textured mesh

In [ ]:
#@title 1. Install (~2 min)
import os, sys, subprocess
M='/content/.tsr_ok3'
if not os.path.exists(M):
    os.chdir('/content')
    subprocess.run(['rm','-rf','/content/2d-to-3d-game-models'])
    subprocess.check_call(['git','clone','-b','claude/image-to-3d-pipeline-CnSII',
        'https://github.com/pmikola/2d-to-3d-game-models.git'])
    subprocess.run(['rm','-rf','/content/TripoSR'])
    subprocess.check_call(['git','clone',
        'https://github.com/VAST-AI-Research/TripoSR.git','/content/TripoSR'])
    # Install only what's needed — skip requirements.txt (breaks numpy)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',
        'transformers','einops','omegaconf','huggingface_hub',
        'trimesh','Pillow','tqdm','rembg','onnxruntime',
        'pygltflib','xatlas','pyyaml','imageio','opencv-python'])
    open(M,'w').write('ok')
    print('Done. Restarting...')
    try:
        import IPython; IPython.get_ipython().kernel.do_shutdown(True)
    except: os._exit(0)
else:
    os.chdir('/content/2d-to-3d-game-models')
    import torch
    if torch.cuda.is_available():
        v=torch.cuda.get_device_properties(0).total_memory/1024**3
        print(f'GPU: {torch.cuda.get_device_name(0)} ({v:.0f}GB) | Ready!')
    import psutil; r=psutil.virtual_memory()
    print(f'RAM: {r.available/1024**3:.1f}GB free')

In [ ]:
#@title 2. Upload image
from google.colab import files
from PIL import Image
from IPython.display import display
import numpy as np
P='/content/input.png'
print('Upload PNG/JPG:')
u=files.upload()
if u:
    import shutil; shutil.copy(list(u.keys())[0],P)
    img=Image.open(P); print(f'{img.size}'); display(img.resize((300,300)))
else:
    a=np.full((512,512,3),220,dtype=np.uint8)
    y,x=np.ogrid[-256:256,-256:256]; a[x**2+y**2<150**2]=[180,80,40]
    Image.fromarray(a).save(P); print('Using test image')

In [ ]:
#@title 3. Generate 3D model
import os, sys, time, shutil, tempfile, base64, gc
import torch, trimesh, numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML

TSR='/content/TripoSR'
REPO='/content/2d-to-3d-game-models'
P='/content/input.png'
OUT='/content/output/model.glb'
RAW='/content/output/raw.glb'
os.makedirs('/content/output',exist_ok=True)

sys.path.insert(0,TSR)
os.chdir(TSR)
t0=time.time()

# --- Load TripoSR ---
print('[1/6] Loading TripoSR (~1GB, fast)...')
from tsr.system import TSR as TripoSRSystem
from tsr.utils import remove_background, resize_foreground

model = TripoSRSystem.from_pretrained(
    'stabilityai/TripoSR',
    config_name='config.yaml',
    weight_name='model.ckpt',
)
model.renderer.set_chunk_size(131072)
model.to('cuda')
print(f'  Loaded in {time.time()-t0:.0f}s | GPU: {torch.cuda.memory_allocated()/1024**3:.1f}GB')

# --- Preprocess ---
print('[2/6] Preprocessing (bg removal + resize)...')
image = Image.open(P)
# Remove background
try:
    image = remove_background(image)
    print('  Background removed')
except Exception as e:
    print(f'  BG removal failed: {e}, using original')
# Resize foreground to fill frame
image = resize_foreground(image, ratio=0.85)
# Convert to RGB on white background
if image.mode == 'RGBA':
    bg = Image.new('RGB', image.size, (255,255,255))
    bg.paste(image, mask=image.split()[3])
    image = bg
display(image.resize((200,200)))

# --- Generate 3D ---
print('[3/6] Generating 3D (~30-60 sec on T4)...')
with torch.no_grad():
    scene_codes = model([image], device='cuda')

# Extract mesh
print('  Extracting mesh...')
meshes = model.extract_mesh(scene_codes, resolution=256)
mesh = meshes[0]  # trimesh.Trimesh
mesh.export(RAW)
print(f'  {len(mesh.vertices)} verts, {len(mesh.faces)} faces')

del model, scene_codes; gc.collect(); torch.cuda.empty_cache()

# --- Repair ---
os.chdir(REPO); sys.path.insert(0,REPO)
print('[4/6] Mesh repair...')
from pipeline.mesh_repair import repair_and_prepare
from pipeline.geometry import normalize_mesh, unwrap_uvs, save_mesh_as_obj
mesh = trimesh.load(RAW, force='mesh')
mesh = repair_and_prepare(mesh, smooth_iterations=3)
normalize_mesh(mesh)
print(f'  {len(mesh.vertices)} verts, {len(mesh.faces)} faces')

# --- UV + PBR ---
print('[5/6] UV + PBR maps...')
unwrap_uvs(mesh)
from pipeline.pbr_maps import generate_pbr_maps, save_pbr_maps
from pipeline.export import export_textured_dir_to_glb, validate_glb
tex=Image.open(P).convert('RGB').resize((1024,1024))
with tempfile.TemporaryDirectory() as tmp:
    op=save_mesh_as_obj(mesh,tmp)
    td=Path(tmp)/'textured'; td.mkdir()
    tex.save(str(td/'texture_atlas.png'))
    shutil.copy(op,str(td/'mesh_textured.obj'))
    pbr=generate_pbr_maps(tex,strength=1.5)
    save_pbr_maps(pbr,str(td/'pbr'))
    r=Image.new('RGB',(256*4,256))
    r.paste(tex.resize((256,256)),(0,0))
    r.paste(pbr['normal'].resize((256,256)),(256,0))
    r.paste(pbr['roughness'].convert('RGB').resize((256,256)),(512,0))
    r.paste(pbr['metallic'].convert('RGB').resize((256,256)),(768,0))
    display(r)
    print('[6/6] Export GLB...')
    export_textured_dir_to_glb(str(td),OUT)

info=validate_glb(OUT)
el=time.time()-t0
sz=os.path.getsize(OUT)/(1024*1024)
print(f'\n{"="*50}')
print(f'DONE in {el:.0f}s | {sz:.1f}MB | {info.get("total_vertices","?")} verts')
print(f'{"="*50}')

with open(OUT,'rb') as f: b=base64.b64encode(f.read()).decode()
display(HTML(f'<h2><a href="data:model/gltf-binary;base64,{b}" download="model.glb" '
    f'style="background:#4CAF50;color:white;padding:15px 30px;'
    f'text-decoration:none;border-radius:8px;font-size:18px;">'
    f'📥 DOWNLOAD model.glb ({sz:.1f}MB)</a></h2>'))
with open(RAW,'rb') as f: br=base64.b64encode(f.read()).decode()
display(HTML(f'<a href="data:model/gltf-binary;base64,{br}" download="raw.glb" '
    f'style="color:#2196F3">Download raw shape</a>'))
print('View: https://gltf-viewer.donmccurdy.com/')